# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata and interface
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a SingleObject, treat as an object
print("Dataset Name:", getattr(metadata, 'name', 'N/A'))
print("Description:", getattr(metadata, 'description', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, data is organized into record sets (`cr:RecordSet`), each with a unique `@id` and set of fields/columns.

In [ ]:
print("Available Record Sets (referenced by '@id'):")
record_sets = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]
if not record_sets:
    # fallback: try to get from the resolved metadata
    record_sets = []
    try:
        resolved = metadata.as_dict() if hasattr(metadata, 'as_dict') else dict(metadata)
        if 'recordSet' in resolved:
            record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in resolved['recordSet']]
    except Exception:
        pass
if not record_sets:
    print("No record sets found in metadata. Dataset may not expose direct record sets.")
else:
    for i, rs_id in enumerate(record_sets):
        print(f"  {i+1}. {rs_id}")

    # Explore the first record set in detail
    rs0_id = record_sets[0]
    rs0 = dataset._get_entity_by_id(rs0_id)
    print(f"\nFields (columns) for Record Set {rs0_id}:")
    fields = rs0.get('field', []) if isinstance(rs0, dict) else []
    if not fields:
        print("  (No fields found)")
    else:
        if isinstance(fields, dict) or isinstance(fields, str):
            fields = [fields]
        for f in fields:
            # f may be an @id or a full dict
            fid = f.get('@id', f) if isinstance(f, dict) else f
            print(f"  - {fid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data for each record set found (usually only one for this dataset)
# If no record sets available, try using fallback for fileObject or similar entities
if not record_sets:
    print("No record sets available to extract.")
    dataframes = {}
else:
    dataframes = {}
    for rsid in record_sets:
        print(f"Extracting data for: {rsid}")
        df = None
        try:
            records = list(dataset.records(record_set=rsid))
            if len(records) > 0:
                df = pd.DataFrame(records)
                dataframes[rsid] = df
            print(f"Columns for {rsid}: {list(df.columns) if df is not None else 'No data.'}")
        except Exception as ex:
            print(f"Failed to extract {rsid}: {ex}")
    # Select one record set to proceed with if at least one dataframe exists
    chosen_rs = record_sets[0]
    df = dataframes.get(chosen_rs)
    if df is not None:
        print(df.head())
    else:
        print(f"No data loaded for {chosen_rs}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Example: Select a numeric field and a group field from the DataFrame
if not record_sets or df is None:
    print("No DataFrame to analyze.")
else:
    # Attempt to auto-detect a numeric field
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        # Try to coerce potential numeric columns
        numeric_fields = []
        for col in df.columns:
            try:
                coerced = pd.to_numeric(df[col], errors='coerce')
                if coerced.notnull().sum() > 0:
                    df[col + "_num"] = coerced
                    numeric_fields.append(col + "_num")
            except Exception:
                continue
    print(f"Numeric fields: {numeric_fields}")

    # Choose first available numeric field, or fallback
    numeric_field = numeric_fields[0] if numeric_fields else None
    group_field = None
    # Try to auto-detect a suitable group/categorical field
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() < 20 and col != numeric_field:
            group_field = col
            break

    if numeric_field:
        threshold = df[numeric_field].quantile(0.75) if df[numeric_field].dtype.kind in 'fi' else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_sets or df is None or not numeric_fields:
    print("No data/numeric fields to visualize.")
else:
    fig, ax = plt.subplots(figsize=(8,4))
    col = numeric_fields[0]
    sns.histplot(df[col].dropna(), kde=True, bins=30, ax=ax)
    ax.set_title(f"Histogram of {col}")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=col, data=df)
        plt.title(f"{col} distribution by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We explored the Mass Spectrometry dataset using the Croissant schema and the `mlcroissant` library.
- The dataset's structure and metadata were inspected, with data loaded and analyzed based on available record sets and fields (all referenced by `@id`).
- Common EDA steps such as filtering and normalization were performed, and visualizations illustrated data distributions.
- This template can be adapted for deeper domain-specific analyses or for automated feature engineering on any Croissant-compliant dataset.